# Healthcare QA Chatbot - BioMistral-7B Evaluation

Evaluate **BioMistral-7B** (medical-specialized) vs **TinyLlama-1.1B** using Colab T4 GPU.

> **Set runtime to GPU first:** Runtime > Change runtime type > T4 GPU


## 1. GPU Check & Install Dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)')


In [ ]:
!pip install -q transformers>=4.36.0 bitsandbytes>=0.41.0 accelerate>=0.25.0 chromadb==0.4.24 sentence-transformers>=2.2.0 rank-bm25>=0.2.2 pandas tqdm
print('Dependencies installed!')


## 2. Upload Knowledge Base

On your local machine first:
```bash
cd ~/Documents/final_project
zip -r knowledge_base.zip data/knowledge_base/
```
Then run the cell below and upload.


In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

KB_DIR = Path('/content/data/knowledge_base')
if KB_DIR.exists() and any(KB_DIR.iterdir()):
    print(f'Knowledge base already at {KB_DIR}')
else:
    print('Upload your knowledge_base.zip...')
    uploaded = files.upload()
    for fname in uploaded:
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/')
    if not KB_DIR.exists():
        for p in Path('/content').rglob('chroma.sqlite3'):
            KB_DIR.mkdir(parents=True, exist_ok=True)
            for item in p.parent.iterdir():
                dest = KB_DIR / item.name
                if item.is_dir(): shutil.copytree(item, dest, dirs_exist_ok=True)
                else: shutil.copy2(item, dest)
            break
    print(f'Extracted to {KB_DIR}: {[f.name for f in KB_DIR.iterdir()]}')


## 3. Initialize Retrieval Pipeline

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path=str(KB_DIR))
collection = client.get_collection('medical_knowledge')
print(f'Knowledge base: {collection.count():,} documents')

def retrieve(query, top_k=5):
    qe = embedder.encode([query])[0].tolist()
    r = collection.query(query_embeddings=[qe], n_results=top_k)
    return [{'content': d, 'source': m.get('source','?'), 'score': round(1-dist,4)}
            for d, m, dist in zip(r['documents'][0], r['metadatas'][0], r['distances'][0])]

test = retrieve('What are the symptoms of diabetes?')
print(f'Test: {len(test)} passages, top score: {test[0]["score"]:.3f}')


## 4. Load Models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

# BioMistral-7B (4-bit quantized)
print('Loading BioMistral-7B with 4-bit quantization...')
t0 = time.time()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
bio_tok = AutoTokenizer.from_pretrained('BioMistral/BioMistral-7B', trust_remote_code=True)
bio_model = AutoModelForCausalLM.from_pretrained(
    'BioMistral/BioMistral-7B', quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True)
bio_model.eval()
if bio_tok.pad_token is None: bio_tok.pad_token = bio_tok.eos_token
print(f'BioMistral loaded in {time.time()-t0:.1f}s, GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB')


In [ ]:
# TinyLlama-1.1B
print('Loading TinyLlama-1.1B...')
t0 = time.time()
tiny_tok = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0', trust_remote_code=True)
tiny_model = AutoModelForCausalLM.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0', torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True)
tiny_model.eval()
if tiny_tok.pad_token is None: tiny_tok.pad_token = tiny_tok.eos_token
print(f'TinyLlama loaded in {time.time()-t0:.1f}s, Total GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB')


## 5. Generation & Cleaning Functions

In [ ]:
import re

STOP_PATTERNS = [
    r'\nQuestion:', r'\nQ:', r'\nAnswer:', r'Best regards', r'Sincerely',
    r'ChatDoctor', r'HealthCareMagic', r'Thank you for', r'Take care',
    r'I hope this', r'\nDear ', r'\n---',
]

def clean(text):
    if not text: return text
    text = text.strip()
    for p in ['Answer:', 'Factual Answer:', 'Based on the reference text,']:
        if text.startswith(p): text = text[len(p):].strip()
    cut = min(50, len(text))
    best = len(text)
    for pat in STOP_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m and m.start() >= cut and m.start() < best: best = m.start()
    return re.sub(r'\[\d+\]', '', text[:best]).strip()

def generate(question, context, model, tokenizer, max_tokens=256):
    """Generate RAG answer. Returns (text, latency_s, n_tokens)."""
    SYS = chr(60) + '|system|' + chr(62)
    USR = chr(60) + '|user|' + chr(62)
    AST = chr(60) + '|assistant|' + chr(62)
    END = chr(60) + '/s' + chr(62)
    prompt = (
        f'{SYS}\n'
        'Answer the question using ONLY the reference text. '
        'Do NOT add your own knowledge. Be concise.\n'
        f'{END}\n'
        f'{USR}\n'
        f'REFERENCE TEXT: {context}\n\n'
        f'QUESTION: {question}\n'
        f'{END}\n'
        f'{AST}\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    input_len = inputs.input_ids.shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.3,
                             top_p=0.85, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    latency = time.time() - t0
    gen_ids = out[0][input_len:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return clean(answer), latency, len(gen_ids)


## 6. Run Evaluation - BioMistral vs TinyLlama

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

EVAL_QUESTIONS = [
    'What are the symptoms of type 2 diabetes?',
    'What causes hypertension?',
    'What is the treatment for asthma?',
    'What are the side effects of metformin?',
    'How is pneumonia diagnosed?',
    'What is the difference between Type 1 and Type 2 diabetes?',
    'What are the risk factors for heart disease?',
    'How does aspirin work as a blood thinner?',
    'What is COPD and how is it treated?',
    'What are the symptoms of a stroke?',
]

results = []

for q in tqdm(EVAL_QUESTIONS, desc='Evaluating'):
    # Retrieve context
    passages = retrieve(q, top_k=5)
    context = '\n\n'.join([p['content'] for p in passages[:3]])
    avg_retrieval_score = sum(p['score'] for p in passages) / len(passages)

    # Generate with BioMistral
    bio_answer, bio_latency, bio_tokens = generate(q, context, bio_model, bio_tok)

    # Generate with TinyLlama
    tiny_answer, tiny_latency, tiny_tokens = generate(q, context, tiny_model, tiny_tok)

    results.append({
        'question': q,
        'retrieval_score': round(avg_retrieval_score, 4),
        'biomistral_answer': bio_answer,
        'biomistral_latency_s': round(bio_latency, 2),
        'biomistral_tokens': bio_tokens,
        'tinyllama_answer': tiny_answer,
        'tinyllama_latency_s': round(tiny_latency, 2),
        'tinyllama_tokens': tiny_tokens,
    })
    print(f'\nQ: {q}')
    print(f'  BioMistral ({bio_latency:.1f}s): {bio_answer[:120]}...')
    print(f'  TinyLlama  ({tiny_latency:.1f}s): {tiny_answer[:120]}...')

df = pd.DataFrame(results)
print(f'\nEvaluation complete! {len(df)} questions evaluated.')


## 7. Results Summary

In [ ]:
# Summary statistics
print('=' * 60)
print('MODEL COMPARISON SUMMARY')
print('=' * 60)
print(f'\nBioMistral-7B:')
print(f'  Avg latency: {df["biomistral_latency_s"].mean():.2f}s')
print(f'  Avg tokens:  {df["biomistral_tokens"].mean():.0f}')
print(f'  Avg answer length: {df["biomistral_answer"].str.len().mean():.0f} chars')
print(f'\nTinyLlama-1.1B:')
print(f'  Avg latency: {df["tinyllama_latency_s"].mean():.2f}s')
print(f'  Avg tokens:  {df["tinyllama_tokens"].mean():.0f}')
print(f'  Avg answer length: {df["tinyllama_answer"].str.len().mean():.0f} chars')
print(f'\nRetrieval:')
print(f'  Avg score: {df["retrieval_score"].mean():.4f}')
print('=' * 60)

# Show full table
df[['question', 'biomistral_latency_s', 'tinyllama_latency_s', 'retrieval_score']]


## 8. Export Results

In [ ]:
# Save as CSV
df.to_csv('/content/biomistral_vs_tinyllama_eval.csv', index=False)
print('Saved to /content/biomistral_vs_tinyllama_eval.csv')

# Download
from google.colab import files
files.download('/content/biomistral_vs_tinyllama_eval.csv')
print('Download started!')


## 9. Side-by-Side Answer Comparison

In [ ]:
# Display all answers side by side
for _, row in df.iterrows():
    print('=' * 80)
    print(f'Q: {row["question"]}')
    print(f'Retrieval Score: {row["retrieval_score"]}')
    print(f'\n--- BioMistral-7B ({row["biomistral_latency_s"]}s) ---')
    print(row['biomistral_answer'])
    print(f'\n--- TinyLlama-1.1B ({row["tinyllama_latency_s"]}s) ---')
    print(row['tinyllama_answer'])
    print()
